# 03 — Train and evaluate

Trains a sklearn `RandomForestRegressor` to predict ultimate incurred from first-report features.
Logs metrics with MLflow and registers the model to Unity Catalog as
`{catalog}.{schema}.ultimate_claim_severity`.

**Requires Dedicated ML Runtime** (`ml_cluster_id`).

### Step 1 — Configure training targets

Set the feature table, Unity Catalog registered model name (`ultimate_claim_severity`), and the Shared MLflow experiment path used to track this run.

In [ ]:
dbutils.widgets.text("catalog", "actuarial")
dbutils.widgets.text("schema", "ml")
dbutils.widgets.text("project_src", "")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
project_src = dbutils.widgets.get("project_src").rstrip("/")

feature_table = f"{catalog}.{schema}.claim_severity_features"
model_name = f"{catalog}.{schema}.ultimate_claim_severity"
experiment_path = f"/Shared/ml-pipeline-demo/ultimate_claim_severity"
print(f"feature_table={feature_table}")
print(f"model_name={model_name}")

### Step 2 — Import shared feature column lists

Add `project_src` to `sys.path` and import `MODEL_FEATURE_COLUMNS`, `NUMERIC_COLUMNS`, and `CATEGORICAL_COLUMNS` from the shared package so training uses the same columns as feature engineering.

In [ ]:
import sys
from pathlib import Path

candidates = [Path(project_src)]
if project_src and not project_src.startswith("/Workspace"):
    candidates.append(Path("/Workspace") / project_src.lstrip("/"))
for p in candidates:
    if p.exists():
        sys.path.insert(0, str(p.resolve()))
        break
else:
    raise FileNotFoundError(f"project_src not found: {project_src}")

from ml_pipeline_demo.features import (
    CATEGORICAL_COLUMNS,
    MODEL_FEATURE_COLUMNS,
    NUMERIC_COLUMNS,
)

### Step 3 — Load data and create a holdout split

Load the feature table into pandas, select model features (`X`) and the ultimate incurred label (`y`), and keep claim/policy metadata for later comparison. Split 80/20 with a fixed random seed for a reproducible holdout set.

In [ ]:
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

pdf = spark.table(feature_table).toPandas()
print(f"Loaded {len(pdf)} training rows")

X = pdf[MODEL_FEATURE_COLUMNS]
y = pdf["ultimate_incurred"]
meta = pdf[["claim_id", "policy_id", "first_incurred", "ultimate_incurred"]]

X_train, X_test, y_train, y_test, meta_train, meta_test = train_test_split(
    X, y, meta, test_size=0.2, random_state=42
)
print(f"train={len(X_train)} test={len(X_test)}")

### Step 4 — Train, evaluate, and register the model

Build a sklearn pipeline (one-hot encode categoricals + `RandomForestRegressor`), fit on the training split, and log MAE / RMSE / R² to MLflow. Register the model in Unity Catalog, preview the worst large-loss holdout rows, and persist a holdout preview table for the batch-score notebook.

In [ ]:
# Pipeline: passthrough numerics, one-hot categoricals, then RandomForest.
preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", NUMERIC_COLUMNS),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            CATEGORICAL_COLUMNS,
        ),
    ]
)

model = Pipeline(
    steps=[
        ("preprocess", preprocess),
        (
            "regressor",
            RandomForestRegressor(
                n_estimators=100,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

mlflow.set_registry_uri("databricks-uc")

# Ensure Workspace parent folder exists for the experiment path.
from databricks.sdk import WorkspaceClient
parent = "/".join(experiment_path.rstrip("/").split("/")[:-1])
if parent:
    WorkspaceClient().workspace.mkdirs(parent)

mlflow.set_experiment(experiment_path)

# Fit, evaluate on holdout, log metrics/params, and register to UC.
with mlflow.start_run(run_name="ultimate_claim_severity_rf") as run:
    mlflow.sklearn.autolog(log_models=False)
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    mae = float(mean_absolute_error(y_test, preds))
    rmse = float(np.sqrt(mean_squared_error(y_test, preds)))
    r2 = float(r2_score(y_test, preds))

    mlflow.log_metric("mae", mae)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)
    mlflow.log_param("n_train", len(X_train))
    mlflow.log_param("n_test", len(X_test))

    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="model",
        registered_model_name=model_name,
        input_example=X_train.head(5),
    )

    print(f"run_id={run.info.run_id}")
    print(f"MAE={mae:,.2f}  RMSE={rmse:,.2f}  R2={r2:.3f}")

# Preview largest actual ultimates: first vs predicted vs actual.
comparison = meta_test.copy()
comparison["predicted_ultimate"] = preds
comparison["error"] = comparison["predicted_ultimate"] - comparison["ultimate_incurred"]
comparison = comparison.sort_values("ultimate_incurred", ascending=False).head(10)
print("Holdout sample: first incurred vs predicted vs actual ultimate")
display(comparison)

# Persist holdout keys for the batch-score notebook demo table.
holdout_table = f"{catalog}.{schema}.claim_severity_holdout"
holdout_pdf = meta_test.copy()
holdout_pdf["predicted_ultimate"] = preds
spark.createDataFrame(holdout_pdf).write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(holdout_table)
print(f"Wrote holdout preview table {holdout_table}")
print("Train + register complete.")